##Scorecard to indentify pertinent fields that need cleaned

In [0]:
# from pyspark.sql import functions as F

# table_name = "virtue_foundation_dataset.virtue_foundation_dataset.facilities"

# df = spark.table(table_name)

# total_rows = df.count()

# results = []

# for field in df.schema.fields:

#     col_name = field.name
#     dtype = field.dataType.simpleString()

#     metrics = {
#         "column_name": col_name,
#         "data_type": dtype
#     }

#     if dtype in ["string"]:

#         row = (
#             df.select(
#                 F.sum(F.when(F.col(col_name).isNull(), 1).otherwise(0)).alias("null_count"),

#                 F.sum(
#                     F.when(
#                         F.trim(F.col(col_name)) == "",
#                         1
#                     ).otherwise(0)
#                 ).alias("blank_count"),

#                 F.sum(
#                     F.when(
#                         F.lower(F.trim(F.col(col_name))) == "null",
#                         1
#                     ).otherwise(0)
#                 ).alias("literal_null_count"),

#                 F.sum(
#                     F.when(
#                         F.col(col_name).rlike(r"^\s*[\[\{].*[\]\}]\s*$"),
#                         1
#                     ).otherwise(0)
#                 ).alias("json_artifact_count"),

#                 F.approx_count_distinct(F.col(col_name)).alias("distinct_count"),

#                 F.min(F.length(F.col(col_name))).alias("min_length"),

#                 F.max(F.length(F.col(col_name))).alias("max_length")
#             )
#             .first()
#         )

#         metrics.update(row.asDict())

#     else:

#         row = (
#             df.select(
#                 F.sum(F.when(F.col(col_name).isNull(), 1).otherwise(0)).alias("null_count"),
#                 F.approx_count_distinct(F.col(col_name)).alias("distinct_count"),
#                 F.min(F.col(col_name)).alias("min_value"),
#                 F.max(F.col(col_name)).alias("max_value")
#             )
#             .first()
#         )

#         metrics.update(row.asDict())

#     metrics["total_rows"] = total_rows
#     metrics["null_pct"] = round(
#         100.0 * metrics.get("null_count", 0) / total_rows, 2
#     )

#     results.append(metrics)

# scorecard_df = spark.createDataFrame(results)

# display(
#     scorecard_df.orderBy(
#         F.desc("null_pct")
#     )
# )

## Scorecard to identify strings that need to be cleaned


In [0]:
# from pyspark.sql import functions as F

# display(
#     scorecard_df
#         .select(
#             "column_name",
#             "data_type",
#             "null_pct",
#             "null_count",
#             "blank_count",
#             "literal_null_count",
#             "json_artifact_count",
#             "distinct_count",
#             "min_length",
#             "max_length",
#             "min_value",
#             "max_value"
#         )
#         .orderBy(
#             F.desc("null_pct"),
#             F.desc("json_artifact_count")
#         )
# )

## Final Cleaning output

In [0]:
# from pyspark.sql import functions as F

# display(
#     spark.table(
#         "virtue_foundation_dataset.virtue_foundation_dataset.facilities"
#     )
#     .select(
#         "unique_id",
#         "name",
#         "organization_type",
#         "address_country",
#         "officialWebsite",
#         "latitude",
#         "longitude"
#     )
#     .filter(
#         F.col("name").isNull()
#         | F.col("latitude").isNull()
#         | F.col("longitude").isNull()
#     )
# )

In [0]:
# display(
#     df.select(
#         "name",
#         "organization_type",
#         "address_country"
#     )
#     .groupBy(
#         "name",
#         "organization_type",
#         "address_country"
#     )
#     .count()
#     .orderBy(F.desc("count"))
# )

## Clean Dataframe Based on Information Revealed Above

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, ArrayType
from pyspark.sql.window import Window

# ------------------------------------------------------------------------------
# Load source data + HARD RESET audit_reason
# ------------------------------------------------------------------------------

source_table = "virtue_foundation_dataset.bronze.facilities"
target_table = "virtue_foundation_dataset.silver.facilities_cleaned"

df = spark.table(source_table)

df = df.drop("audit_reason") if "audit_reason" in df.columns else df

df = df.withColumn(
    "audit_reason",
    F.lit(None).cast(StringType())
)

# ------------------------------------------------------------------------------
# Helper
# ------------------------------------------------------------------------------

def set_reason(condition, reason):
    return F.when(condition, F.lit(reason)).otherwise(F.col("audit_reason"))

# ------------------------------------------------------------------------------
# Steps 1–4: validations
# ------------------------------------------------------------------------------

valid_uuid = F.col("unique_id").rlike("^[a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12}$")

df = df.withColumn("audit_reason", set_reason(~valid_uuid, "invalid_uuid_format"))

valid_name = (
    F.col("name").isNotNull()
    & (F.trim(F.col("name")) != "")
    & ~F.lower(F.trim(F.col("name"))).isin("null", "[]", "{}")
)

df = df.withColumn("audit_reason", set_reason(~valid_name, "invalid_name"))

valid_org = F.lower(F.trim(F.col("organization_type"))) == "facility"

df = df.withColumn("audit_reason", set_reason(~valid_org, "invalid_org_type"))

valid_coords = (
    F.col("latitude").isNull()
    | F.col("longitude").isNull()
    | (
        F.col("latitude").between(6.0, 37.6)
        & F.col("longitude").between(68.0, 97.5)
    )
)

df = df.withColumn("audit_reason", set_reason(~valid_coords, "invalid_coordinates_or_outside_india"))

# ------------------------------------------------------------------------------
# Step 5–7: cleaning
# ------------------------------------------------------------------------------

df = df.withColumn("address_zipOrPostCode", F.regexp_replace(F.col("address_zipOrPostCode"), r"\s+", ""))

string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

for c in string_cols:
    df = df.withColumn(c, F.lower(F.trim(F.col(c))))

# ------------------------------------------------------------------------------
# Step 8: DUPLICATE DETECTION (MOVED UP)
# ------------------------------------------------------------------------------

geo_window = Window.partitionBy("name", "address_country", "latitude", "longitude")

address_window = Window.partitionBy("name", "address_country", "address_zipOrPostCode", "address_line1")

df = df.withColumn(
    "audit_reason",
    F.when(F.count("*").over(geo_window) > 1, F.lit("duplicate_record"))
    .otherwise(F.col("audit_reason"))
)

df = df.withColumn(
    "audit_reason",
    F.when(F.count("*").over(address_window) > 1, F.lit("duplicate_record"))
    .otherwise(F.col("audit_reason"))
)

# ------------------------------------------------------------------------------
# Step 7: SAFE LIST NORMALIZATION (DO NOT TOUCH ADDRESSES)
# ------------------------------------------------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType

def clean_string_list(col):
    return F.array_distinct(
        F.split(
            F.regexp_replace(F.col(col), r"[\[\]\"']+", ""),
            r"\s*,\s*"
        )
    )

array_cols = []
string_cols_for_lists = []

for f in df.schema.fields:
    if isinstance(f.dataType, ArrayType):
        array_cols.append(f.name)
    elif isinstance(f.dataType, StringType):
        string_cols_for_lists.append(f.name)

# Clean real arrays
for c in array_cols:
    df = df.withColumn(
        c,
        F.array_distinct(F.expr(f"filter({c}, x -> x is not null)"))
    )

# ------------------------------------------------------------
# DO NOT SPLIT ADDRESS FIELDS
# ------------------------------------------------------------
EXCLUDE_FROM_SPLIT = {
    "address_line1",
    "address_line2",
    "address_city",
    "address_state",
    "address_country",
    "address_zipOrPostCode"
}

for c in string_cols_for_lists:
    if c != "audit_reason" and c not in EXCLUDE_FROM_SPLIT:
        df = df.withColumn(
            c,
            F.when(
                F.col(c).rlike(r"^\[.*\]$"),   # ONLY bracketed lists
                clean_string_list(c)
            ).otherwise(
                F.array(F.col(c))
            )
        )

# ------------------------------------------------------------------------------
# STEP 7.5: COLUMN TYPE DECISION BASED ONLY ON CLEAN ROWS
# ------------------------------------------------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType

array_cols = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, ArrayType)
]

clean_df = df.filter(F.col("audit_reason").isNull())

def collapse_or_keep_column(df, col_name):

    # ------------------------------------------------------------
    # SCHEMA INFERENCE ONLY FROM CLEAN ROWS
    # ------------------------------------------------------------
    max_size = clean_df.select(
        F.max(F.size(F.col(col_name)))
    ).collect()[0][0]

    # ------------------------------------------------------------
    # CASE 1: scalar-only in clean data → collapse to STRING
    # ------------------------------------------------------------
    if max_size is None or max_size <= 1:

        df = df.withColumn(
            col_name,
            F.when(
                F.size(F.col(col_name)) == 0,
                F.lit(None).cast("string")
            ).when(
                F.size(F.col(col_name)) == 1,
                F.element_at(F.col(col_name), 1).cast("string")
            ).otherwise(
                F.lit(None).cast("string")
            )
        )

    # ------------------------------------------------------------
    # CASE 2: true array column → keep ARRAY
    # ------------------------------------------------------------
    else:

        df = df.withColumn(
            col_name,
            F.expr(f"filter({col_name}, x -> x is not null)")
        )

    return df


for c in array_cols:
    df = collapse_or_keep_column(df, c)

# ------------------------------------------------------------------------------
# WRITE
# ------------------------------------------------------------------------------

df.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(target_table)